In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
!pip install pyarrow==8.0.0  # Downgrade pyarrow to a known compatible version
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from datasets import load_dataset, Dataset
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 17.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 24.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 46.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 16.1.0 which

In [ ]:


# Step 2: Define Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name_or_path = 'google/muril-base-cased'
tokenizer_name_or_path = 'google/muril-base-cased'
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128
lr = 3e-2
num_epochs = 50
batch_size = 8

# Step 3: Load and Prepare Dataset
column_names = [label_column, "blank", text_column]


#1% - 52, 2%- 104, 10% - 520
number = 6
dataset_train = pd.read_csv("/content/drive/MyDrive/train_dataset_Manipuri.csv",encoding = 'utf-8', header=None, names=column_names).head(number)

dataset_test = pd.read_csv("/content/drive/MyDrive/test_dataset_Manipuri.csv",encoding = 'utf-8', header=None, names=column_names)

dataset_train.drop("blank", axis=1, inplace=True)
dataset_train[label_column] = dataset_train[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset_train.head())

dataset_test.drop("blank", axis=1, inplace=True)
dataset_test[label_column] = dataset_test[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset_test.head())


# from sklearn.model_selection import train_test_split
# train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
dataset_train = Dataset.from_pandas(dataset_train)
dataset_test = Dataset.from_pandas(dataset_test)

# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Step 5: Preprocess Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], truncation=True, max_length=max_length, padding="max_length")
    model_inputs["labels"] = examples[label_column]
    return model_inputs

# Step 6: Apply Preprocessing
processed_datasets_train = dataset_train.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_train.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)
processed_datasets_test = dataset_test.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_test.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

# Step 7: DataLoaders
train_dataloader = DataLoader(
    processed_datasets_train, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
test_dataloader = DataLoader(
    processed_datasets_test, shuffle=False, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)

# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)
model = get_peft_model(model, peft_config)
model.dropout = torch.nn.Dropout(0.1)  # Add dropout to prevent overfitting
print(model.print_trainable_parameters())

# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

# Step 10: Training Loop
  model = model.to(device)
  for epoch in range(num_epochs):
      model.train()
      total_loss = 0
      for step, batch in enumerate(tqdm(train_dataloader)):
          batch = {k: v.to(device) for k, v in batch.items()}
          outputs = model(**batch)
          loss = outputs.loss
          total_loss += loss.detach().float()
          loss.backward()
          optimizer.step()
          lr_scheduler.step()
          optimizer.zero_grad()

    # model.eval()
    # eval_loss = 0
    # eval_preds = []
    # true_labels = []
    # for step, batch in enumerate(tqdm(test_dataloader)):
    #     batch = {k: v.to(device) for k, v in batch.items()}
    #     with torch.no_grad():
    #         outputs = model(**batch)
    #     loss = outputs.loss
    #     eval_loss += loss.detach().float()
    #     eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())
    #     true_labels.extend(batch["labels"].detach().cpu().numpy())

    # eval_epoch_loss = eval_loss / len(test_dataloader)
    train_epoch_loss = total_loss / len(train_dataloader)
    # accuracy = accuracy_score(true_labels, eval_preds)
    # f1 = f1_score(true_labels, eval_preds, average='weighted')
    # recall = recall_score(true_labels, eval_preds, average='weighted')
    # precision = precision_score(true_labels, eval_preds, average='weighted')

    print(f"Epoch {epoch}: train_loss={train_epoch_loss}")#, eval_loss={eval_epoch_loss}, accuracy={accuracy}, f1_score={f1}, recall={recall}, precision={precision}


   text_label                                         Tweet text
0           1  ꯑꯄꯨꯅꯕꯥ ꯑꯣꯏꯅꯥ ꯃꯁꯤꯒꯤ ꯄꯔꯐꯣꯃꯦꯟꯁ ꯑꯁꯤ ꯌꯥꯝꯅꯥ ꯐꯩ ꯍꯥꯌꯅꯥ...
1           1  ꯑꯅꯨꯄꯝ ꯈꯦꯔꯒꯤ ꯑꯈꯟꯅꯕꯥ ꯑꯦꯛꯇꯤꯡ ꯁ꯭ꯇꯥꯏꯜ ꯑꯁꯤ ꯃꯐꯝ ꯑꯁꯤꯗꯥ...
2           1                     USB ꯍꯣꯁ꯭ꯠ ꯁꯄꯣꯔꯠ ꯑꯃꯁꯨꯡ 1GB ꯔꯦꯝ꯫
3           1  ꯔꯘꯨꯒꯤ ꯍꯣꯡꯂꯛꯂꯤꯕꯥ ꯏꯃꯣꯁꯅꯁꯤꯡ ꯑꯗꯨ ꯐꯣꯡꯗꯣꯛꯅꯕꯒꯤꯗꯃꯛ ꯃꯍꯥ...
4           1  ꯍꯥꯌꯕꯗꯤ ꯃꯁꯤ ꯁ꯭ꯃꯥꯔꯇꯐꯣꯟ ꯈꯛꯇꯅꯤ, ꯑꯗꯨꯕꯨ ꯀꯔꯤꯒꯨꯝꯕꯥ ꯅꯍꯥ...
   text_label                                         Tweet text
0           1  ꯑꯄꯨꯅꯕꯥ ꯑꯣꯏꯅꯥ ꯃꯁꯤꯒꯤ ꯄꯔꯐꯣꯃꯦꯟꯁ ꯑꯁꯤ ꯌꯥꯝꯅꯥ ꯐꯩ ꯍꯥꯌꯅꯥ...
1           0  ꯕꯦꯠꯇꯔꯤ ꯑꯁꯤ ꯈꯔꯥ ꯋꯥꯠꯄꯅꯥ ꯃꯔꯝ ꯑꯣꯏꯗꯨꯅꯥ ꯊꯕꯛ ꯇꯧꯔꯤꯕꯥ ꯃ...
2           1            ꯑꯦꯞ ꯑꯃꯥ ꯂꯧꯊꯣꯀꯄꯥ ꯑꯁꯤꯁꯨ ꯌꯥꯝꯅꯥ ꯂꯥꯏꯕꯛ ꯐꯕꯅꯤ꯫
3           1  ꯅꯣꯔꯃꯦꯜ ꯑꯣꯏꯅꯥ ꯁꯤꯖꯤꯟꯅꯕꯗꯥ, ꯃꯁꯤꯒꯤ ꯕꯦꯠꯇꯔꯤ ꯑꯁꯤ ꯋꯥꯏ-ꯐ...
4           0                   ꯃꯁꯤꯒꯤ ꯃꯃꯂꯒꯥ ꯃꯥꯟꯅꯕꯥ ꯐꯤꯆꯔꯁꯤꯡ ꯂꯩꯇꯦ꯫


Running tokenizer on dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/2401 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 7,682 || all params: 237,565,444 || trainable%: 0.0032
None


...

Epoch 47: train_loss=0.0026288072112947702


100%|██████████| 1/1 [00:00<00:00, 41.91it/s]


Epoch 48: train_loss=0.0024945540353655815


100%|██████████| 1/1 [00:00<00:00, 38.08it/s]

Epoch 49: train_loss=0.0025720165576785803


In [ ]:
    model.eval()
    eval_loss = 0
    eval_preds = []
    true_labels = []
    for step, batch in enumerate(tqdm(test_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())
        true_labels.extend(batch['labels'].detach().cpu().numpy())

    print(f"Epoch {epoch}: Number of predictions = {len(eval_preds)}, Number of true labels = {len(true_labels)}")

100%|██████████| 301/301 [00:19<00:00, 15.12it/s]

Epoch 49: Number of predictions = 2401, Number of true labels = 2401


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [ ]:
accuracy = accuracy_score(true_labels, eval_preds)
f1 = f1_score(true_labels, eval_preds, average='weighted')
recall = recall_score(true_labels, eval_preds, average='weighted')
precision = precision_score(true_labels, eval_preds, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

Accuracy: 0.7592669720949604
F1 Score: 0.655371065402042
Recall: 0.7592669720949604
Precision: 0.5764863349142494


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# After training, output the predictions
# Convert predictions and true labels to a DataFrame for easy analysis
results_df = pd.DataFrame({
    'true_label': true_labels,
    'predicted_label': eval_preds
})
print(results_df)

      true_label  predicted_label
0              1                1
1              0                1
2              1                1
3              1                1
4              0                1
...          ...              ...
2396           0                1
2397           1                1
2398           0                1
2399           1                1
2400           1                1

[2401 rows x 2 columns]


In [ ]:
# Save the results to a CSV file
results_df.to_csv("/content/drive/MyDrive/model_predictions.csv", index=False)
print("Predictions saved to /content/drive/MyDrive/model_predictions.csv")

# Or print the first few rows of the results
print(results_df.head())

Predictions saved to /content/drive/MyDrive/model_predictions.csv
   true_label  predicted_label
0           0                0
1           1                0
2           0                0
3           1                1
4           0                1
